[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/14_kv_cache.ipynb)

# 🔴 Hard: KV Cache Attention

Implement **multi-head attention with KV caching** for efficient autoregressive generation.

During LLM inference, recomputing all key/value projections at every step is wasteful.
A **KV cache** stores previously computed K and V tensors so only the new token(s) need projection.

### Signature
```python
class KVCacheAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, cache=None) -> tuple[torch.Tensor, tuple]:
        # x: (B, S_new, D) — new tokens
        # cache: None or (K_past, V_past) each (B, num_heads, S_past, d_k)
        # Returns: (output, (K_all, V_all))
```

### Requirements
- Inherit from `nn.Module`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` projections
- When `cache=None` (prefill): apply **causal mask**, return all K/V as cache
- When `cache` provided (decode): concat new K/V with cached, no causal mask needed for single-token decode
- Incremental decode must produce **identical** results to full forward pass

### Key Idea
```
Prefill:  [t0 t1 t2 t3] → full causal attention → cache = (K_{0:3}, V_{0:3})
Decode:   [t4]           → Q=t4, K/V=cache+t4  → cache = (K_{0:4}, V_{0:4})
Decode:   [t5]           → Q=t5, K/V=cache+t5  → cache = (K_{0:5}, V_{0:5})
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn as nn
import math

/usr/local/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        # Initialize W_q, W_k, W_v, W_o
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.d_model = d_model
        self.num_heads = num_heads

    def forward(self, x, cache=None):
        batch_size, seq_len, _ = x.shape
        d_head = self.d_model // self.num_heads
        # 1. Project Q, K, V from x
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        # 2. Reshape to multi-head: (B, num_heads, S, d_k)
        Q = Q.view(batch_size, seq_len, self.num_heads, d_head).transpose(1,2)
        K = K.view(batch_size, seq_len, self.num_heads, d_head).transpose(1,2)
        V = V.view(batch_size, seq_len, self.num_heads, d_head).transpose(1,2)
        # 3. If cache exists, concat new K/V with cached K/V
        if cache:
            K = torch.cat([cache[0], K], dim = 2)
            V = torch.cat([cache[1], V], dim = 2)
        # 4. Compute attention (causal mask needed during prefill)
        atten_score = Q @ K.transpose(-1, -2) # we only require the context value for x (the input)
        total_seq_len = K.shape[2]
        mask = torch.tril(torch.ones(seq_len, total_seq_len), total_seq_len - seq_len) # the last token must attend to all tokens
        atten_score = atten_score.masked_fill(mask == 0, float('-inf'))
        atten_score = torch.softmax(atten_score / math.sqrt(d_head), dim = -1)
        context_vec = atten_score @ V
        context_vec = context_vec.transpose(1,2)
        context_vec = context_vec.contiguous().view(batch_size, seq_len, self.d_model)
        output = self.W_o(context_vec)
        # 5. Return (output, (K_all, V_all))
        return output, (K,V)

In [6]:
# 🧪 Debug
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 6, 64)

# Full forward
full_out, _ = attn(x)
print("Full output shape:", full_out.shape)  # (1, 6, 64)

# Incremental: prefill 4, decode 1, decode 1
out1, cache = attn(x[:, :4])
print("Cache K shape:", cache[0].shape)  # (1, 4, 4, 16)
out2, cache = attn(x[:, 4:5], cache=cache)
out3, cache = attn(x[:, 5:6], cache=cache)
inc_out = torch.cat([out1, out2, out3], dim=1)
print("Match:", torch.allclose(full_out, inc_out, atol=1e-5))

Full output shape: torch.Size([1, 6, 64])
Cache K shape: torch.Size([1, 4, 4, 16])
Match: True


In [7]:
# ✅ SUBMIT
from torch_judge import check
check('kv_cache')


🧪 Testing: KV Cache Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (no cache) (4.4ms)
  ✅ [2/5] Cache structure (1.5ms)
  ✅ [3/5] Decode step appends to cache (1.4ms)
  ✅ [4/5] Incremental decode matches full forward (2.3ms)
  ✅ [5/5] Gradient flow (12.3ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (21.8ms total)
  Progress saved. Run status() to see your dashboard.

